# Home Credit Proxy Notebook 03 (Rewritten)

Notebook này build **proxy feature mart** cho 3 stage A/B/C bằng **Polars**, theo hướng:

- đọc output từ notebook 1 và 2
- chọn variable pools theo business proxy + shortlist
- aggregate raw train CSV về `case_id`
- build proxy blocks A / B / C
- map stage labels **động** theo output thực tế của notebook 2
- export:
  - `proxy_feature_mart.parquet`
  - `proxy_features_A.parquet`
  - `proxy_features_B.parquet`
  - `proxy_features_C.parquet`
  - `proxy_feature_registry.csv`
  - `missingness_*.csv`

Điểm sửa chính so với bản cũ:
- bỏ hardcode stage labels gây filter rỗng
- stage subsets được tạo từ **prefix thực tế** của `stage_final`
- code aggregate và chọn proxy feature an toàn hơn
- output review rõ hơn để kiểm tra sẵn sàng cho notebook 4/5


In [1]:

from __future__ import annotations

import gc
import re
from pathlib import Path
from typing import Sequence

import polars as pl

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(140)
pl.Config.set_tbl_width_chars(260)


polars.config.Config

In [2]:

# =========================================================
# 1. Paths
# =========================================================
TRAIN_DIR = Path("/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train")
if not TRAIN_DIR.exists():
    for p in [
        Path("/kaggle/input/home-credit-credit-risk-model-stability/csv_files/train"),
        Path("/kaggle/input/homecredit-credit-risk-model-stability/csv_files/train"),
    ]:
        if p.exists():
            TRAIN_DIR = p
            break

INPUT_DIR_NB1 = Path("/kaggle/input/notebooks/hongvittrnh/homecredit-eda-part1/homecredit_proxy_notebook_01")
INPUT_DIR_NB2 = Path("/kaggle/input/notebooks/hongvittrnh/homecredit-eda-part2/homecredit_proxy_notebook_02")

if not INPUT_DIR_NB1.exists():
    for p in [
        Path("/kaggle/working/homecredit_proxy_notebook_01"),
        Path("/kaggle/working"),
        Path("/mnt/data"),
        Path.cwd(),
    ]:
        if (p / "feature_catalog.parquet").exists() and (p / "header_inventory.parquet").exists():
            INPUT_DIR_NB1 = p
            break

if not INPUT_DIR_NB2.exists():
    for p in [
        Path("/kaggle/working/homecredit_proxy_notebook_02"),
        Path("/kaggle/working"),
        Path("/mnt/data"),
        Path.cwd(),
    ]:
        if (p / "stage_assignment_proxy.parquet").exists() and (p / "proxy_eda_shortlist.parquet").exists():
            INPUT_DIR_NB2 = p
            break

WORK_DIR = Path("/kaggle/working/homecredit_proxy_notebook_03")
WORK_DIR.mkdir(parents=True, exist_ok=True)

TEMP_DIR = WORK_DIR / "agg_parts"
TEMP_DIR.mkdir(parents=True, exist_ok=True)

STAGE_ASSIGN_PATH = INPUT_DIR_NB2 / "stage_assignment_proxy.parquet"
PROXY_SHORTLIST_PATH = INPUT_DIR_NB2 / "proxy_eda_shortlist.parquet"

FEATURE_CATALOG_PATH = INPUT_DIR_NB1 / "feature_catalog.parquet"
HEADER_INV_PATH = INPUT_DIR_NB1 / "header_inventory.parquet"

required_paths = [
    STAGE_ASSIGN_PATH,
    PROXY_SHORTLIST_PATH,
    FEATURE_CATALOG_PATH,
    HEADER_INV_PATH,
]

for p in required_paths:
    assert p.exists(), f"Missing required file: {p}"

print("TRAIN_DIR    :", TRAIN_DIR)
print("INPUT_DIR_NB1:", INPUT_DIR_NB1)
print("INPUT_DIR_NB2:", INPUT_DIR_NB2)
print("WORK_DIR     :", WORK_DIR)


TRAIN_DIR    : /kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train
INPUT_DIR_NB1: /kaggle/input/notebooks/hongvittrnh/homecredit-eda-part1/homecredit_proxy_notebook_01
INPUT_DIR_NB2: /kaggle/input/notebooks/hongvittrnh/homecredit-eda-part2/homecredit_proxy_notebook_02
WORK_DIR     : /kaggle/working/homecredit_proxy_notebook_03


In [3]:

# =========================================================
# 2. Load upstream outputs
# =========================================================
stage_assignment = pl.read_parquet(STAGE_ASSIGN_PATH)
proxy_shortlist = pl.read_parquet(PROXY_SHORTLIST_PATH)
feature_catalog = pl.read_parquet(FEATURE_CATALOG_PATH)
header_inventory = pl.read_parquet(HEADER_INV_PATH)

print("stage_assignment:", stage_assignment.shape)
print("proxy_shortlist :", proxy_shortlist.shape)
print("feature_catalog :", feature_catalog.shape)
print("header_inventory:", header_inventory.shape)


stage_assignment: (1526659, 15)
proxy_shortlist : (276, 19)
feature_catalog : (472, 25)
header_inventory: (1139, 4)


In [4]:

# =========================================================
# 3. Helpers
# =========================================================
CASE_COL = "case_id"
DECISION_COL = "decision_date"
TARGET_COL = "target"

BASE_META_COLS = [CASE_COL, DECISION_COL, TARGET_COL, "tenure_months_proxy", "anchor_quality_flag", "stage_final"]

def normalize_colname(c: str) -> str:
    return re.sub(r"[^a-z0-9_]+", "", str(c).strip().lower())

def read_csv_header(path: Path) -> list[str]:
    with open(path, "r", encoding="utf-8") as f:
        first = f.readline().strip("\n\r")
    return first.split(",")

def cast_numeric_to_f32(df: pl.DataFrame, exclude: set[str] | None = None) -> pl.DataFrame:
    exclude = exclude or set()
    exprs = []
    numeric_dtypes = {
        pl.Float64, pl.Float32,
        pl.Int64, pl.Int32, pl.Int16, pl.Int8,
        pl.UInt64, pl.UInt32, pl.UInt16, pl.UInt8,
    }
    for c, dt in zip(df.columns, df.dtypes):
        if c in exclude:
            continue
        if dt in numeric_dtypes and dt != pl.Float32:
            exprs.append(pl.col(c).cast(pl.Float32))
    return df.with_columns(exprs) if exprs else df

def safe_ratio_expr(num_col: str, den_col: str, out_col: str) -> pl.Expr:
    return (
        pl.when(
            pl.col(num_col).is_null()
            | pl.col(den_col).is_null()
            | (pl.col(den_col) == 0)
        )
        .then(None)
        .otherwise(pl.col(num_col) / pl.col(den_col))
        .cast(pl.Float32)
        .alias(out_col)
    )

def with_bool_flag(df: pl.DataFrame, expr: pl.Expr, out_col: str) -> pl.DataFrame:
    return df.with_columns(expr.cast(pl.Int8).alias(out_col))

def choose_existing(df: pl.DataFrame, cols: list[str], out_col: str, how: str = "max") -> pl.DataFrame:
    cols = [c for c in cols if c in df.columns]
    if not cols:
        return df.with_columns(pl.lit(None).cast(pl.Float32).alias(out_col))

    if how == "max":
        expr = pl.max_horizontal(cols)
    elif how == "min":
        expr = pl.min_horizontal(cols)
    elif how == "sum":
        expr = pl.sum_horizontal(cols)
    else:
        raise ValueError(f"Unsupported how={how}")

    return df.with_columns(expr.cast(pl.Float32).alias(out_col))

def find_feature_cols(df: pl.DataFrame, base_var: str, contains: str | None = None) -> list[str]:
    base_var = normalize_colname(base_var)
    cols = [c for c in df.columns if base_var in normalize_colname(c)]
    if contains is not None:
        cols = [c for c in cols if contains in c]
    return cols

def add_proxy_from_raw(
    block: pl.DataFrame,
    raw_df: pl.DataFrame,
    base_var: str,
    out_col: str,
    contains: str = "__max",
    how: str = "max",
) -> pl.DataFrame:
    cols = find_feature_cols(raw_df, base_var, contains=contains)
    if not cols:
        return block.with_columns(pl.lit(None).cast(pl.Float32).alias(out_col))
    add_df = raw_df.select([CASE_COL] + cols)
    return choose_existing(block.join(add_df, on=CASE_COL, how="left"), cols, out_col, how=how)

def null_rate(df: pl.DataFrame, col_name: str) -> float:
    if df.height == 0:
        return 1.0
    return float(df[col_name].null_count() / df.height)

def stage_prefix_from_value(val: str | None) -> str:
    val = (val or "").strip().lower()
    if val.startswith("a"):
        return "A"
    if val.startswith("b"):
        return "B"
    if val.startswith("c"):
        return "C"
    return "OTHER"


In [5]:

# =========================================================
# 4. Raw header inventory from train files
# =========================================================
train_files = sorted(TRAIN_DIR.glob("*.csv"))
assert len(train_files) > 0, f"No csv files found under {TRAIN_DIR}"

RAW_HEADERS = {p.name: read_csv_header(p) for p in train_files}
FILE_HEADERS = {p.name: [normalize_colname(c) for c in RAW_HEADERS[p.name]] for p in train_files}
HEADER_MAP = {p.name: {normalize_colname(c): c for c in RAW_HEADERS[p.name]} for p in train_files}
FILE_PATH_MAP = {p.name: p for p in train_files}

print("train_files:", len(train_files))
print("sample files:", [p.name for p in train_files[:8]])


train_files: 32
sample files: ['train_applprev_1_0.csv', 'train_applprev_1_1.csv', 'train_applprev_2.csv', 'train_base.csv', 'train_credit_bureau_a_1_0.csv', 'train_credit_bureau_a_1_1.csv', 'train_credit_bureau_a_1_2.csv', 'train_credit_bureau_a_1_3.csv']


In [6]:

# =========================================================
# 5. Stage base + dynamic stage mapping
# =========================================================
keep_cols = [c for c in BASE_META_COLS if c in stage_assignment.columns]
stage_base = (
    stage_assignment
    .select(keep_cols)
    .unique(subset=[CASE_COL], keep="first")
)

if DECISION_COL in stage_base.columns and stage_base[DECISION_COL].dtype != pl.Date:
    try:
        stage_base = stage_base.with_columns(pl.col(DECISION_COL).str.to_date(strict=False))
    except Exception:
        pass

stage_base = stage_base.with_columns(
    pl.col("stage_final").cast(pl.Utf8).fill_null("UNKNOWN").alias("stage_final"),
    pl.col("stage_final").map_elements(stage_prefix_from_value, return_dtype=pl.Utf8).alias("stage_group")
)

stage_summary = (
    stage_base
    .group_by(["stage_group", "stage_final"])
    .agg(pl.len().alias("n_cases"))
    .sort(["stage_group", "n_cases"], descending=[False, True])
)

print("stage_base:", stage_base.shape)
display(stage_summary)
display(stage_base.head(5))


stage_base: (1526659, 6)


stage_group,stage_final,n_cases
str,str,u32
"""A""","""A_new_no_internal_history""",318617
"""B""","""B_behavior_no_collection""",1085030
"""C""","""C_collection_signal""",123012


case_id,decision_date,target,tenure_months_proxy,stage_final,stage_group
i64,date,i64,f64,str,str
873780,2019-11-24,0,null,"""A_new_no_internal_history""","""A"""
1483698,2019-08-10,0,27.53,"""B_behavior_no_collection""","""B"""
1538766,2019-09-15,0,20.44,"""B_behavior_no_collection""","""B"""
1395913,2019-06-09,0,5.29,"""B_behavior_no_collection""","""B"""
1680461,2019-12-13,0,null,"""B_behavior_no_collection""","""B"""


In [7]:

# =========================================================
# 6. Seed variables by business meaning
# =========================================================
SEED_A = [
    "amtdepositincoming_4809444A",
    "amtdepositbalance_4809441A",
    "amount_4527230A",
    "amount_4917619A",
    "contractssum_5085716L",
    "credquantity_1099L",
    "amount_1115A",
    "applicationcnt_361L",
]

SEED_B = [
    "avgmaxdpdlast9m_3716943P",
    "actualdpd_943P",
    "amtinstpaidbefduel24m_4187115A",
    "annuity_853A",
]

SEED_C = [
    "avgdbddpdlast24m_3658932P",
    "avgdpdtolclosure24_3658938P",
    "actualdpd_943P",
    "datelastinstal40dpd_247D",
]

SEED_A = [normalize_colname(x) for x in SEED_A]
SEED_B = [normalize_colname(x) for x in SEED_B]
SEED_C = [normalize_colname(x) for x in SEED_C]

print("SEED_A:", SEED_A)
print("SEED_B:", SEED_B)
print("SEED_C:", SEED_C)


SEED_A: ['amtdepositincoming_4809444a', 'amtdepositbalance_4809441a', 'amount_4527230a', 'amount_4917619a', 'contractssum_5085716l', 'credquantity_1099l', 'amount_1115a', 'applicationcnt_361l']
SEED_B: ['avgmaxdpdlast9m_3716943p', 'actualdpd_943p', 'amtinstpaidbefduel24m_4187115a', 'annuity_853a']
SEED_C: ['avgdbddpdlast24m_3658932p', 'avgdpdtolclosure24_3658938p', 'actualdpd_943p', 'datelastinstal40dpd_247d']


In [8]:

# =========================================================
# 7. Build variable pools from shortlist + seeds
# =========================================================
proxy_df = proxy_shortlist.with_columns([
    pl.col("variable").cast(pl.Utf8).str.to_lowercase().alias("variable_norm"),
    pl.col("proxy_concept_final").cast(pl.Utf8).fill_null("").str.to_lowercase().alias("proxy_concept_final_norm"),
    pl.col("stage_candidate_final").cast(pl.Utf8).fill_null("").str.to_lowercase().alias("stage_candidate_final_norm"),
    pl.col("feature_family").cast(pl.Utf8).fill_null("").str.to_lowercase().alias("feature_family_norm"),
])

def vars_for_stage(seed_vars: list[str], stage_key: str) -> list[str]:
    stage_key = stage_key.lower()

    if stage_key == "a":
        filt = (
            pl.col("stage_candidate_final_norm").str.contains("a")
            | pl.col("proxy_concept_final_norm").str.contains("a_")
            | pl.col("feature_family_norm").is_in([
                "capacity_income_liquidity",
                "external_bureau_exposure",
                "fraud_instability",
                "demographic_stability",
            ])
        )
    elif stage_key == "b":
        filt = (
            pl.col("stage_candidate_final_norm").str.contains("b")
            | pl.col("proxy_concept_final_norm").str.contains("b_")
            | pl.col("feature_family_norm").is_in([
                "delinquency",
                "prepayment_willingness",
                "installment_burden",
            ])
        )
    elif stage_key == "c":
        filt = (
            pl.col("stage_candidate_final_norm").str.contains("c")
            | pl.col("proxy_concept_final_norm").str.contains("c_")
            | pl.col("feature_family_norm").is_in([
                "delinquency",
                "serious_delinquency",
                "roll_rate_collection",
            ])
        )
    else:
        raise ValueError(stage_key)

    shortlist_vars = (
        proxy_df
        .filter(filt)
        .select("variable_norm")
        .unique()
        .to_series()
        .to_list()
    )

    out = sorted(set(shortlist_vars).union(seed_vars))
    return out

A_VARS = vars_for_stage(SEED_A, "a")
B_VARS = vars_for_stage(SEED_B, "b")
C_VARS = vars_for_stage(SEED_C, "c")
ALL_VARS = sorted(set(A_VARS).union(B_VARS).union(C_VARS))

print("A_VARS:", len(A_VARS))
print("B_VARS:", len(B_VARS))
print("C_VARS:", len(C_VARS))
print("ALL_VARS:", len(ALL_VARS))


A_VARS: 144
B_VARS: 210
C_VARS: 67
ALL_VARS: 212


In [9]:

# =========================================================
# 8. Relevant files for selected variables
# =========================================================
header_norm = header_inventory.with_columns(
    pl.col("column_name").cast(pl.Utf8).str.to_lowercase().alias("variable_norm")
)

def files_for_vars(vars_list: list[str]) -> list[str]:
    if not vars_list:
        return []
    return (
        header_norm
        .filter(pl.col("variable_norm").is_in(vars_list))
        .select("file_name")
        .unique()
        .sort("file_name")
        .to_series()
        .to_list()
    )

A_FILES = files_for_vars(A_VARS)
B_FILES = files_for_vars(B_VARS)
C_FILES = files_for_vars(C_VARS)
ALL_FILES = sorted(set(A_FILES).union(B_FILES).union(C_FILES))

print("A_FILES:", len(A_FILES), A_FILES[:10])
print("B_FILES:", len(B_FILES), B_FILES[:10])
print("C_FILES:", len(C_FILES), C_FILES[:10])
print("ALL_FILES:", len(ALL_FILES))
assert len(ALL_FILES) > 0, "No relevant files selected."


A_FILES: 28 ['train_applprev_1_0.csv', 'train_applprev_1_1.csv', 'train_applprev_2.csv', 'train_credit_bureau_a_1_0.csv', 'train_credit_bureau_a_1_1.csv', 'train_credit_bureau_a_1_2.csv', 'train_credit_bureau_a_1_3.csv', 'train_credit_bureau_a_2_0.csv', 'train_credit_bureau_a_2_1.csv', 'train_credit_bureau_a_2_10.csv']
B_FILES: 29 ['train_applprev_1_0.csv', 'train_applprev_1_1.csv', 'train_applprev_2.csv', 'train_credit_bureau_a_1_0.csv', 'train_credit_bureau_a_1_1.csv', 'train_credit_bureau_a_1_2.csv', 'train_credit_bureau_a_1_3.csv', 'train_credit_bureau_a_2_0.csv', 'train_credit_bureau_a_2_1.csv', 'train_credit_bureau_a_2_10.csv']
C_FILES: 22 ['train_applprev_1_0.csv', 'train_applprev_1_1.csv', 'train_credit_bureau_a_1_0.csv', 'train_credit_bureau_a_1_1.csv', 'train_credit_bureau_a_1_2.csv', 'train_credit_bureau_a_1_3.csv', 'train_credit_bureau_a_2_0.csv', 'train_credit_bureau_a_2_1.csv', 'train_credit_bureau_a_2_10.csv', 'train_credit_bureau_a_2_2.csv']
ALL_FILES: 29


In [10]:

# =========================================================
# 9. Safe CSV reader
# =========================================================
def safe_read_selected_columns(path: Path, variables_norm: Sequence[str]) -> pl.DataFrame | None:
    header_norm = FILE_HEADERS[path.name]
    header_map = HEADER_MAP[path.name]

    cols_norm = [normalize_colname(c) for c in variables_norm if normalize_colname(c) in header_norm]
    if CASE_COL in header_norm and CASE_COL not in cols_norm:
        cols_norm = [CASE_COL] + cols_norm

    cols_norm = list(dict.fromkeys(cols_norm))
    if CASE_COL not in cols_norm or len(cols_norm) <= 1:
        return None

    cols_raw = [header_map[c] for c in cols_norm]

    try:
        df = pl.read_csv(
            path,
            columns=cols_raw,
            infer_schema_length=3000,
            low_memory=True,
            ignore_errors=True,
            try_parse_dates=False,
        )
        rename_map = {raw: normalize_colname(raw) for raw in cols_raw}
        return df.rename(rename_map)
    except Exception as e:
        print(f"[WARN] Failed reading {path.name}: {e}")
        return None


In [11]:

# =========================================================
# 10. Aggregate one file to case level
# =========================================================
suffix_lookup = (
    feature_catalog
    .with_columns(pl.col("variable").cast(pl.Utf8).str.to_lowercase().alias("variable_norm"))
    .select(["variable_norm", "suffix_type"])
    .unique()
)
SUFFIX_MAP = dict(zip(suffix_lookup["variable_norm"].to_list(), suffix_lookup["suffix_type"].to_list()))

def aggregate_file_to_case_level(file_name: str, variables_norm: Sequence[str]) -> pl.DataFrame | None:
    path = FILE_PATH_MAP[file_name]
    df = safe_read_selected_columns(path, variables_norm)
    if df is None or CASE_COL not in df.columns:
        return None

    keep = [c for c in df.columns if c == CASE_COL or c in variables_norm]
    df = df.select(keep)

    agg_exprs = []
    for col_name in df.columns:
        if col_name == CASE_COL:
            continue

        suffix_type = SUFFIX_MAP.get(col_name, "other")

        if suffix_type == "date" or col_name.endswith("d"):
            parsed = False
            try:
                df = df.with_columns(pl.col(col_name).str.to_date(strict=False))
                if df[col_name].dtype == pl.Date:
                    parsed = True
            except Exception:
                parsed = False

            if parsed:
                agg_exprs += [
                    pl.col(col_name).min().alias(f"{file_name}__{col_name}__min_date"),
                    pl.col(col_name).max().alias(f"{file_name}__{col_name}__max_date"),
                ]
            else:
                agg_exprs += [
                    pl.col(col_name).cast(pl.Utf8, strict=False).drop_nulls().first().alias(f"{file_name}__{col_name}__first_txt"),
                ]

        elif suffix_type in {"amount", "count", "dpd", "numeric"} or col_name.endswith(("a", "l", "p", "t")):
            agg_exprs += [
                pl.col(col_name).cast(pl.Float64, strict=False).max().alias(f"{file_name}__{col_name}__max"),
                pl.col(col_name).cast(pl.Float64, strict=False).mean().alias(f"{file_name}__{col_name}__mean"),
                pl.col(col_name).cast(pl.Float64, strict=False).sum().alias(f"{file_name}__{col_name}__sum"),
            ]

        else:
            agg_exprs += [
                pl.col(col_name).cast(pl.Utf8, strict=False).drop_nulls().first().alias(f"{file_name}__{col_name}__first_txt"),
                pl.col(col_name).cast(pl.Utf8, strict=False).n_unique().alias(f"{file_name}__{col_name}__nunique"),
            ]

    if not agg_exprs:
        return None

    out = df.group_by(CASE_COL).agg(agg_exprs)
    return cast_numeric_to_f32(out, exclude={CASE_COL})


In [12]:

# =========================================================
# 11. Aggregate relevant files to temp parquet
# =========================================================
temp_parts = []
for file_name in ALL_FILES:
    part_path = TEMP_DIR / f"{file_name.replace('.csv', '')}_agg.parquet"

    if part_path.exists():
        temp_parts.append(part_path)
        print(f"[CACHE] {file_name} -> {part_path.name}")
        continue

    part = aggregate_file_to_case_level(file_name, ALL_VARS)
    if part is None:
        print(f"[SKIP] {file_name}")
        continue

    part = part.unique(subset=[CASE_COL], keep="first")
    part.write_parquet(part_path)
    temp_parts.append(part_path)
    print(f"[SAVED] {file_name}: {part.shape} -> {part_path.name}")

    del part
    gc.collect()

assert len(temp_parts) > 0, "No aggregated parts were created."
print("temp_parts:", len(temp_parts))
print([p.name for p in temp_parts[:10]])


[SAVED] train_applprev_1_0.csv: (782997, 75) -> train_applprev_1_0_agg.parquet
[SAVED] train_applprev_1_1.csv: (438525, 75) -> train_applprev_1_1_agg.parquet
[SAVED] train_applprev_2.csv: (1221522, 7) -> train_applprev_2_agg.parquet
[SAVED] train_credit_bureau_a_1_0.csv: (335275, 88) -> train_credit_bureau_a_1_0_agg.parquet
[SAVED] train_credit_bureau_a_1_1.csv: (549263, 88) -> train_credit_bureau_a_1_1_agg.parquet
[SAVED] train_credit_bureau_a_1_2.csv: (325127, 88) -> train_credit_bureau_a_1_2_agg.parquet
[SAVED] train_credit_bureau_a_1_3.csv: (176608, 88) -> train_credit_bureau_a_1_3_agg.parquet
[SAVED] train_credit_bureau_a_2_0.csv: (98303, 19) -> train_credit_bureau_a_2_0_agg.parquet
[SAVED] train_credit_bureau_a_2_1.csv: (118481, 19) -> train_credit_bureau_a_2_1_agg.parquet
[SAVED] train_credit_bureau_a_2_10.csv: (23734, 19) -> train_credit_bureau_a_2_10_agg.parquet
[SAVED] train_credit_bureau_a_2_2.csv: (156749, 19) -> train_credit_bureau_a_2_2_agg.parquet
[SAVED] train_credit_bu

In [13]:

# =========================================================
# 12. Build raw feature mart
# =========================================================
base_cols = [c for c in BASE_META_COLS + ["stage_group"] if c in stage_base.columns]
raw_feature_mart = stage_base.select(base_cols)

for path in temp_parts:
    part = pl.read_parquet(path)
    raw_feature_mart = raw_feature_mart.join(part, on=CASE_COL, how="left")
    print(f"[JOINED] {path.name}: {part.shape}")
    del part
    gc.collect()

raw_feature_mart = cast_numeric_to_f32(raw_feature_mart, exclude={CASE_COL})
print("raw_feature_mart:", raw_feature_mart.shape)
display(raw_feature_mart.head(3))


[JOINED] train_applprev_1_0_agg.parquet: (782997, 75)
[JOINED] train_applprev_1_1_agg.parquet: (438525, 75)
[JOINED] train_applprev_2_agg.parquet: (1221522, 7)
[JOINED] train_credit_bureau_a_1_0_agg.parquet: (335275, 88)
[JOINED] train_credit_bureau_a_1_1_agg.parquet: (549263, 88)
[JOINED] train_credit_bureau_a_1_2_agg.parquet: (325127, 88)
[JOINED] train_credit_bureau_a_1_3_agg.parquet: (176608, 88)
[JOINED] train_credit_bureau_a_2_0_agg.parquet: (98303, 19)
[JOINED] train_credit_bureau_a_2_1_agg.parquet: (118481, 19)
[JOINED] train_credit_bureau_a_2_10_agg.parquet: (23734, 19)
[JOINED] train_credit_bureau_a_2_2_agg.parquet: (156749, 19)
[JOINED] train_credit_bureau_a_2_3_agg.parquet: (190486, 19)
[JOINED] train_credit_bureau_a_2_4_agg.parquet: (190313, 19)
[JOINED] train_credit_bureau_a_2_5_agg.parquet: (231250, 19)
[JOINED] train_credit_bureau_a_2_6_agg.parquet: (150426, 19)
[JOINED] train_credit_bureau_a_2_7_agg.parquet: (45056, 19)
[JOINED] train_credit_bureau_a_2_8_agg.parquet: (

case_id,decision_date,target,tenure_months_proxy,stage_final,stage_group,train_applprev_1_0.csv__actualdpd_943p__max,train_applprev_1_0.csv__actualdpd_943p__mean,train_applprev_1_0.csv__actualdpd_943p__sum,train_applprev_1_0.csv__annuity_853a__max,…,train_static_cb_0.csv__forweek_601l__sum,train_static_cb_0.csv__foryear_618l__max,train_static_cb_0.csv__foryear_618l__mean,train_static_cb_0.csv__foryear_618l__sum,train_tax_registry_a_1.csv__amount_4527230a__max,train_tax_registry_a_1.csv__amount_4527230a__mean,train_tax_registry_a_1.csv__amount_4527230a__sum,train_tax_registry_b_1.csv__amount_4917619a__max,train_tax_registry_b_1.csv__amount_4917619a__mean,train_tax_registry_b_1.csv__amount_4917619a__sum
i64,date,f32,f32,str,str,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
873780,2019-11-24,0.0,null,"""A_new_no_internal_history""","""A""",null,null,null,null,…,0.0,null,null,0.0,5137.399902,3060.628662,21424.400391,null,null,null
1483698,2019-08-10,0.0,27.530001,"""B_behavior_no_collection""","""B""",0.0,0.0,0.0,8947.600586,…,0.0,null,null,0.0,null,null,null,null,null,null
1538766,2019-09-15,0.0,20.440001,"""B_behavior_no_collection""","""B""",0.0,0.0,0.0,7298.200195,…,0.0,null,null,0.0,2415.0,1893.072754,20823.800781,null,null,null


In [14]:

# =========================================================
# 13. Build A proxy block
# =========================================================
A_block = raw_feature_mart.select([c for c in BASE_META_COLS + ["stage_group"] if c in raw_feature_mart.columns])

A_block = add_proxy_from_raw(A_block, raw_feature_mart, "amtdepositincoming_4809444a", "a_amtdepositincoming_max")
A_block = add_proxy_from_raw(A_block, raw_feature_mart, "amtdepositbalance_4809441a", "a_amtdepositbalance_max")
A_block = add_proxy_from_raw(A_block, raw_feature_mart, "amount_4527230a", "a_tax_amount_4527230_max")
A_block = add_proxy_from_raw(A_block, raw_feature_mart, "amount_4917619a", "a_other_amount_4917619_max")
A_block = add_proxy_from_raw(A_block, raw_feature_mart, "contractssum_5085716l", "a_contractssum_max")
A_block = add_proxy_from_raw(A_block, raw_feature_mart, "credquantity_1099l", "a_credquantity_max")
A_block = add_proxy_from_raw(A_block, raw_feature_mart, "amount_1115a", "a_amount_1115_max")
A_block = add_proxy_from_raw(A_block, raw_feature_mart, "applicationcnt_361l", "a_applicationcnt_max")

A_block = A_block.with_columns([
    safe_ratio_expr("a_contractssum_max", "a_amtdepositincoming_max", "a_external_debt_to_incoming_ratio"),
    safe_ratio_expr("a_amtdepositbalance_max", "a_amtdepositincoming_max", "a_deposit_balance_to_incoming_ratio"),
])

A_block = with_bool_flag(
    A_block,
    (pl.col("a_applicationcnt_max").fill_null(0) >= 2),
    "a_has_multi_application_signal"
)

A_block = with_bool_flag(
    A_block,
    (pl.col("a_credquantity_max").fill_null(0) > 0),
    "a_has_external_credit_exposure"
)

print("A_block:", A_block.shape)
display(A_block.select([c for c in A_block.columns if c.startswith("a_")][:20]).head(3))


A_block: (1526659, 27)


a_amtdepositincoming_max,a_amtdepositbalance_max,a_tax_amount_4527230_max,a_other_amount_4917619_max,a_contractssum_max,a_credquantity_max,a_amount_1115_max,a_applicationcnt_max,a_external_debt_to_incoming_ratio,a_deposit_balance_to_incoming_ratio,a_has_multi_application_signal,a_has_external_credit_exposure
f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i8,i8
null,null,5137.399902,null,null,null,null,0.0,null,null,0,0
null,null,null,null,null,null,null,0.0,null,null,0,0
null,null,2415.0,null,null,null,null,0.0,null,null,0,0


In [15]:

# =========================================================
# 14. Build B proxy block
# =========================================================
B_block = raw_feature_mart.select([c for c in BASE_META_COLS + ["stage_group"] if c in raw_feature_mart.columns])

B_block = add_proxy_from_raw(B_block, raw_feature_mart, "actualdpd_943p", "b_actualdpd_max")
B_block = add_proxy_from_raw(B_block, raw_feature_mart, "avgmaxdpdlast9m_3716943p", "b_avgmaxdpdlast9m_max")
B_block = add_proxy_from_raw(B_block, raw_feature_mart, "amtinstpaidbefduel24m_4187115a", "b_amtinstpaidbefdue_max")
B_block = add_proxy_from_raw(B_block, raw_feature_mart, "annuity_853a", "b_annuity_max")

B_block = B_block.with_columns([
    safe_ratio_expr("b_amtinstpaidbefdue_max", "b_annuity_max", "b_paid_before_due_to_annuity_ratio"),
])

B_block = with_bool_flag(
    B_block,
    (pl.col("b_actualdpd_max").fill_null(0) > 0) | (pl.col("b_avgmaxdpdlast9m_max").fill_null(0) > 0),
    "b_has_recent_dpd"
)

B_block = with_bool_flag(
    B_block,
    pl.col("b_amtinstpaidbefdue_max").fill_null(0) > 0,
    "b_has_paid_before_due_signal"
)

print("B_block:", B_block.shape)
display(B_block.select([c for c in B_block.columns if c.startswith("b_")][:20]).head(3))


B_block: (1526659, 21)


b_actualdpd_max,b_avgmaxdpdlast9m_max,b_amtinstpaidbefdue_max,b_annuity_max,b_paid_before_due_to_annuity_ratio,b_has_recent_dpd,b_has_paid_before_due_signal
f32,f32,f32,f32,f32,i8,i8
null,null,null,null,null,0,0
0.0,null,21013.400391,8947.600586,2.348495,0,1
0.0,0.0,145964.0,7298.200195,20.0,0,1


In [16]:

# =========================================================
# 15. Build C proxy block
# =========================================================
C_block = raw_feature_mart.select([c for c in BASE_META_COLS + ["stage_group"] if c in raw_feature_mart.columns])

C_block = add_proxy_from_raw(C_block, raw_feature_mart, "avgdbddpdlast24m_3658932p", "c_avgdbddpdlast24m_max")
C_block = add_proxy_from_raw(C_block, raw_feature_mart, "avgdpdtolclosure24_3658938p", "c_avgdpdtolclosure24_max")
C_block = add_proxy_from_raw(C_block, raw_feature_mart, "actualdpd_943p", "c_actualdpd_max")

date_cols = find_feature_cols(raw_feature_mart, "datelastinstal40dpd_247d", contains="__max_date")
if date_cols and DECISION_COL in C_block.columns:
    date_src = raw_feature_mart.select([CASE_COL] + date_cols)
    C_block = C_block.join(date_src, on=CASE_COL, how="left")

    recency_exprs = []
    for c in date_cols:
        if c in C_block.columns:
            recency_exprs.append((pl.col(DECISION_COL) - pl.col(c)).dt.total_days())

    if recency_exprs:
        C_block = C_block.with_columns(
            pl.min_horizontal(recency_exprs).cast(pl.Float32).alias("c_days_since_last_40dpd")
        )
    else:
        C_block = C_block.with_columns(pl.lit(None).cast(pl.Float32).alias("c_days_since_last_40dpd"))
else:
    C_block = C_block.with_columns(pl.lit(None).cast(pl.Float32).alias("c_days_since_last_40dpd"))

C_block = C_block.with_columns([
    safe_ratio_expr("c_avgdbddpdlast24m_max", "c_avgdpdtolclosure24_max", "c_recent_vs_long_dpd_ratio"),
])

C_block = with_bool_flag(
    C_block,
    (pl.col("c_actualdpd_max").fill_null(0) > 0) | (pl.col("c_avgdbddpdlast24m_max").fill_null(0) > 0),
    "c_has_active_delinquency_signal"
)

C_block = with_bool_flag(
    C_block,
    pl.col("c_days_since_last_40dpd").is_not_null() & (pl.col("c_days_since_last_40dpd") <= 180),
    "c_recent_40dpd_flag"
)

print("C_block:", C_block.shape)
display(C_block.select([c for c in C_block.columns if c.startswith("c_")][:20]).head(3))


C_block: (1526659, 21)


c_avgdbddpdlast24m_max,c_avgdpdtolclosure24_max,c_actualdpd_max,c_days_since_last_40dpd,c_recent_vs_long_dpd_ratio,c_has_active_delinquency_signal,c_recent_40dpd_flag
f32,f32,f32,f32,f32,i8,i8
null,null,null,null,null,0,0
-2.0,0.0,0.0,null,null,0,0
-2.0,0.0,0.0,null,null,0,0


In [17]:

# =========================================================
# 16. Merge final proxy mart + dynamic stage subsets
# =========================================================
drop_overlap = [c for c in BASE_META_COLS + ["stage_group"] if c != CASE_COL]

proxy_feature_mart = (
    stage_base
    .join(A_block.drop([c for c in drop_overlap if c in A_block.columns]), on=CASE_COL, how="left")
    .join(B_block.drop([c for c in drop_overlap if c in B_block.columns]), on=CASE_COL, how="left")
    .join(C_block.drop([c for c in drop_overlap if c in C_block.columns]), on=CASE_COL, how="left")
)

proxy_feature_mart = cast_numeric_to_f32(proxy_feature_mart, exclude={CASE_COL})

A_dataset = proxy_feature_mart.filter(pl.col("stage_group") == "A")
B_dataset = proxy_feature_mart.filter(pl.col("stage_group") == "B")
C_dataset = proxy_feature_mart.filter(pl.col("stage_group") == "C")

print("proxy_feature_mart:", proxy_feature_mart.shape)
print("A_dataset:", A_dataset.shape)
print("B_dataset:", B_dataset.shape)
print("C_dataset:", C_dataset.shape)

display(
    proxy_feature_mart
    .group_by(["stage_group", "stage_final"])
    .agg(pl.len().alias("n_cases"))
    .sort(["stage_group", "n_cases"], descending=[False, True])
)
display(proxy_feature_mart.head(3))


proxy_feature_mart: (1526659, 57)
A_dataset: (318617, 57)
B_dataset: (1085030, 57)
C_dataset: (123012, 57)


stage_group,stage_final,n_cases
str,str,u32
"""A""","""A_new_no_internal_history""",318617
"""B""","""B_behavior_no_collection""",1085030
"""C""","""C_collection_signal""",123012


case_id,decision_date,target,tenure_months_proxy,stage_final,stage_group,train_other_1.csv__amtdepositincoming_4809444a__max,a_amtdepositincoming_max,train_other_1.csv__amtdepositbalance_4809441a__max,a_amtdepositbalance_max,…,c_avgdpdtolclosure24_max,train_applprev_1_0.csv__actualdpd_943p__max_right,train_applprev_1_1.csv__actualdpd_943p__max_right,c_actualdpd_max,train_static_0_0.csv__datelastinstal40dpd_247d__max_date,train_static_0_1.csv__datelastinstal40dpd_247d__max_date,c_days_since_last_40dpd,c_recent_vs_long_dpd_ratio,c_has_active_delinquency_signal,c_recent_40dpd_flag
i64,date,f32,f32,str,str,f32,f32,f32,f32,…,f32,f32,f32,f32,date,date,f32,f32,f32,f32
873780,2019-11-24,0.0,null,"""A_new_no_internal_history""","""A""",null,null,null,null,…,null,null,null,null,null,null,null,null,0.0,0.0
1483698,2019-08-10,0.0,27.530001,"""B_behavior_no_collection""","""B""",null,null,null,null,…,0.0,0.0,null,0.0,null,null,null,null,0.0,0.0
1538766,2019-09-15,0.0,20.440001,"""B_behavior_no_collection""","""B""",null,null,null,null,…,0.0,0.0,null,0.0,null,null,null,null,0.0,0.0


In [18]:

# =========================================================
# 17. Feature registry
# =========================================================
registry_rows = []

for col_name in proxy_feature_mart.columns:
    if col_name in BASE_META_COLS + ["stage_group"]:
        continue

    if col_name.startswith("a_"):
        stage = "A"
    elif col_name.startswith("b_"):
        stage = "B"
    elif col_name.startswith("c_"):
        stage = "C"
    else:
        stage = "OTHER"

    registry_rows.append({
        "feature_name": col_name,
        "stage": stage,
        "dtype": str(proxy_feature_mart[col_name].dtype),
        "non_null_rate": 1.0 - null_rate(proxy_feature_mart, col_name),
    })

feature_registry = (
    pl.DataFrame(registry_rows)
    .sort(["stage", "feature_name"])
    if registry_rows
    else pl.DataFrame({"feature_name": [], "stage": [], "dtype": [], "non_null_rate": []})
)

print("feature_registry:", feature_registry.shape)
display(feature_registry)


feature_registry: (51, 4)


feature_name,stage,dtype,non_null_rate
str,str,str,f64
"""a_amount_1115_max""","""A""","""Float32""",0.016662
"""a_amtdepositbalance_max""","""A""","""Float32""",0.033478
"""a_amtdepositincoming_max""","""A""","""Float32""",0.033478
"""a_applicationcnt_max""","""A""","""Float32""",1.0
"""a_contractssum_max""","""A""","""Float32""",0.103054
"""a_credquantity_max""","""A""","""Float32""",0.023874
"""a_deposit_balance_to_incoming_ratio""","""A""","""Float32""",0.003356
"""a_external_debt_to_incoming_ratio""","""A""","""Float32""",0.001405
"""a_has_external_credit_exposure""","""A""","""Float32""",1.0


In [19]:

# =========================================================
# 18. Missingness summaries by stage block
# =========================================================
def missingness_table(df: pl.DataFrame, prefix: str) -> pl.DataFrame:
    cols = [c for c in df.columns if c.startswith(prefix)]
    rows = []
    for c in cols:
        rows.append({
            "feature_name": c,
            "missing_rate": null_rate(df, c),
            "non_null_rate": 1.0 - null_rate(df, c),
        })
    return (
        pl.DataFrame(rows).sort("missing_rate", descending=True)
        if rows else
        pl.DataFrame({"feature_name": [], "missing_rate": [], "non_null_rate": []})
    )

missing_A = missingness_table(A_dataset, "a_")
missing_B = missingness_table(B_dataset, "b_")
missing_C = missingness_table(C_dataset, "c_")

print("missing_A:", missing_A.shape)
print("missing_B:", missing_B.shape)
print("missing_C:", missing_C.shape)

display(missing_A)
display(missing_B)
display(missing_C)


missing_A: (12, 3)
missing_B: (7, 3)
missing_C: (7, 3)


feature_name,missing_rate,non_null_rate
str,f64,f64
"""a_external_debt_to_incoming_ratio""",0.999934,0.000066
"""a_deposit_balance_to_incoming_ratio""",0.999664,0.000336
"""a_amtdepositincoming_max""",0.998126,0.001874
"""a_amtdepositbalance_max""",0.998126,0.001874
"""a_amount_1115_max""",0.992298,0.007702
"""a_credquantity_max""",0.981278,0.018722
"""a_contractssum_max""",0.947081,0.052919
"""a_other_amount_4917619_max""",0.925387,0.074613
"""a_tax_amount_4527230_max""",0.694724,0.305276


feature_name,missing_rate,non_null_rate
str,f64,f64
"""b_avgmaxdpdlast9m_max""",0.369459,0.630541
"""b_paid_before_due_to_annuity_ratio""",0.227069,0.772931
"""b_amtinstpaidbefdue_max""",0.212267,0.787733
"""b_annuity_max""",0.002454,0.997546
"""b_actualdpd_max""",0.000004,0.999996
"""b_has_recent_dpd""",0.0,1.0
"""b_has_paid_before_due_signal""",0.0,1.0


feature_name,missing_rate,non_null_rate
str,f64,f64
"""c_days_since_last_40dpd""",0.605095,0.394905
"""c_recent_vs_long_dpd_ratio""",0.11442,0.88558
"""c_avgdpdtolclosure24_max""",0.000431,0.999569
"""c_avgdbddpdlast24m_max""",0.000024,0.999976
"""c_actualdpd_max""",0.0,1.0
"""c_has_active_delinquency_signal""",0.0,1.0
"""c_recent_40dpd_flag""",0.0,1.0


In [20]:

# =========================================================
# 19. Business review snapshot
# =========================================================
review_cols = [
    c for c in [
        "stage_group",
        "stage_final",
        "a_external_debt_to_incoming_ratio",
        "a_deposit_balance_to_incoming_ratio",
        "a_has_multi_application_signal",
        "a_has_external_credit_exposure",
        "b_has_recent_dpd",
        "b_has_paid_before_due_signal",
        "b_paid_before_due_to_annuity_ratio",
        "c_has_active_delinquency_signal",
        "c_recent_40dpd_flag",
        "c_recent_vs_long_dpd_ratio",
        "c_days_since_last_40dpd",
    ]
    if c in proxy_feature_mart.columns
]

if review_cols:
    business_review = proxy_feature_mart.select(review_cols)
    print("business_review:", business_review.shape)
    display(business_review.head(20))
else:
    business_review = pl.DataFrame()
    print("No business review columns available.")


business_review: (1526659, 13)


stage_group,stage_final,a_external_debt_to_incoming_ratio,a_deposit_balance_to_incoming_ratio,a_has_multi_application_signal,a_has_external_credit_exposure,b_has_recent_dpd,b_has_paid_before_due_signal,b_paid_before_due_to_annuity_ratio,c_has_active_delinquency_signal,c_recent_40dpd_flag,c_recent_vs_long_dpd_ratio,c_days_since_last_40dpd
str,str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""A""","""A_new_no_internal_history""",null,null,0.0,0.0,0.0,0.0,null,0.0,0.0,null,null
"""B""","""B_behavior_no_collection""",null,null,0.0,0.0,0.0,1.0,2.348495,0.0,0.0,null,null
"""B""","""B_behavior_no_collection""",null,null,0.0,0.0,0.0,1.0,20.0,0.0,0.0,null,null
"""B""","""B_behavior_no_collection""",null,null,0.0,0.0,0.0,1.0,5.0,0.0,0.0,null,null
"""B""","""B_behavior_no_collection""",null,null,0.0,0.0,0.0,1.0,18.985535,0.0,0.0,null,null
"""B""","""B_behavior_no_collection""",null,null,0.0,0.0,1.0,1.0,29.071678,0.0,0.0,null,null
"""A""","""A_new_no_internal_history""",null,null,0.0,0.0,0.0,0.0,null,0.0,0.0,null,null
"""A""","""A_new_no_internal_history""",null,null,0.0,0.0,0.0,0.0,null,0.0,0.0,null,null
"""B""","""B_behavior_no_collection""",null,null,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null


In [21]:

# =========================================================
# 20. Readiness checks before notebook 4 / 5
# =========================================================
stage_ready = pl.DataFrame([
    {
        "stage": "A",
        "n_rows": A_dataset.height,
        "n_features": len([c for c in A_dataset.columns if c.startswith("a_")]),
        "has_target": TARGET_COL in A_dataset.columns,
    },
    {
        "stage": "B",
        "n_rows": B_dataset.height,
        "n_features": len([c for c in B_dataset.columns if c.startswith("b_")]),
        "has_target": TARGET_COL in B_dataset.columns,
    },
    {
        "stage": "C",
        "n_rows": C_dataset.height,
        "n_features": len([c for c in C_dataset.columns if c.startswith("c_")]),
        "has_target": TARGET_COL in C_dataset.columns,
    },
]).with_columns(
    ((pl.col("n_rows") > 0) & (pl.col("n_features") > 0) & pl.col("has_target")).alias("ready_for_modeling")
)

display(stage_ready)

if not stage_ready["ready_for_modeling"].all():
    print("[WARN] Some stage datasets are not ready. Review stage mapping / variable coverage before notebook 4.")
else:
    print("[OK] All stage datasets look usable for the next step.")


stage,n_rows,n_features,has_target,ready_for_modeling
str,i64,i64,bool,bool
"""A""",318617,12,true,true
"""B""",1085030,7,true,true
"""C""",123012,7,true,true


[OK] All stage datasets look usable for the next step.


In [22]:

# =========================================================
# 21. Export outputs
# =========================================================
proxy_mart_path = WORK_DIR / "proxy_feature_mart.parquet"
A_path = WORK_DIR / "proxy_features_A.parquet"
B_path = WORK_DIR / "proxy_features_B.parquet"
C_path = WORK_DIR / "proxy_features_C.parquet"
registry_path = WORK_DIR / "proxy_feature_registry.csv"
missing_A_path = WORK_DIR / "missingness_A.csv"
missing_B_path = WORK_DIR / "missingness_B.csv"
missing_C_path = WORK_DIR / "missingness_C.csv"
stage_summary_path = WORK_DIR / "stage_summary_proxy.csv"
readiness_path = WORK_DIR / "stage_readiness_proxy.csv"

proxy_feature_mart.write_parquet(proxy_mart_path)
A_dataset.write_parquet(A_path)
B_dataset.write_parquet(B_path)
C_dataset.write_parquet(C_path)
feature_registry.write_csv(registry_path)
missing_A.write_csv(missing_A_path)
missing_B.write_csv(missing_B_path)
missing_C.write_csv(missing_C_path)
stage_summary.write_csv(stage_summary_path)
stage_ready.write_csv(readiness_path)

print("Saved outputs:")
for p in [
    proxy_mart_path, A_path, B_path, C_path,
    registry_path, missing_A_path, missing_B_path, missing_C_path,
    stage_summary_path, readiness_path
]:
    print("-", p)


Saved outputs:
- /kaggle/working/homecredit_proxy_notebook_03/proxy_feature_mart.parquet
- /kaggle/working/homecredit_proxy_notebook_03/proxy_features_A.parquet
- /kaggle/working/homecredit_proxy_notebook_03/proxy_features_B.parquet
- /kaggle/working/homecredit_proxy_notebook_03/proxy_features_C.parquet
- /kaggle/working/homecredit_proxy_notebook_03/proxy_feature_registry.csv
- /kaggle/working/homecredit_proxy_notebook_03/missingness_A.csv
- /kaggle/working/homecredit_proxy_notebook_03/missingness_B.csv
- /kaggle/working/homecredit_proxy_notebook_03/missingness_C.csv
- /kaggle/working/homecredit_proxy_notebook_03/stage_summary_proxy.csv
- /kaggle/working/homecredit_proxy_notebook_03/stage_readiness_proxy.csv
